In [1]:
import pandas as pd
import numpy as np
import re
import os
from datasets import Dataset
import evaluate
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer

# 1. 基础配置
model_path = "roberta-base"
tokenizer = AutoTokenizer.from_pretrained(model_path)
metric = evaluate.load("accuracy")

# 2. 数据加载与清洗 (延续之前的特征工程)
df = pd.read_csv('data/train.csv')

def clean_text(text):
    text = str(text).lower()
    text = re.sub(r'http\S+|www\S+|https\S+', '', text, flags=re.MULTILINE)
    text = re.sub(r'\s+', ' ', text)
    return text.strip()

def preprocess_df(input_df):
    temp_df = input_df.copy()
    temp_df['keyword'] = temp_df['keyword'].fillna('unknown')
    temp_df['text'] = temp_df['text'].apply(clean_text)
    # 拼接核心特征
    temp_df['text'] = "keyword: " + temp_df['keyword'].apply(clean_text) + ", text: " + temp_df['text']
    return temp_df.drop(['location', 'keyword'], axis=1, errors='ignore')

train_df_processed = preprocess_df(df)
dataset = Dataset.from_pandas(train_df_processed)

config.json:   0%|          | 0.00/481 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [2]:
# 3. Tokenization (DeBERTa 建议 128 长度)
def tokenize_function(examples):
    return tokenizer(examples["text"], padding="max_length", truncation=True, max_length=128)

tokenized_dataset = dataset.map(tokenize_function, batched=True)

# 4. 准备训练集和验证集
def transform_dataset(ds):
    ds = ds.remove_columns(["id", "text"])
    ds = ds.rename_column("target", "labels")
    return ds

final_dataset = transform_dataset(tokenized_dataset).train_test_split(test_size=0.1, seed=42)

Map:   0%|          | 0/7613 [00:00<?, ? examples/s]

In [3]:
# 5. 模型初始化与指标计算
model = AutoModelForSequenceClassification.from_pretrained(model_path, num_labels=2)

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    return metric.compute(predictions=predictions, references=labels)

# 修改第 6 部分的训练参数
training_args = TrainingArguments(
    output_dir="roberta_output",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,               # RoBERTa 的经典稳定学习率
    per_device_train_batch_size=16,   
    per_device_eval_batch_size=16,
    num_train_epochs=3,               # 3 轮通常就能达到很好的效果
    weight_decay=0.01,                
    warmup_ratio=0.1,                 # 使用比例预热，更自然
    load_best_model_at_end=True,
    metric_for_best_model="accuracy",
    bf16=True,                        # 重新开启 bf16，加速训练
    report_to="none"
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=final_dataset["train"],
    eval_dataset=final_dataset["test"],
    compute_metrics=compute_metrics,
)

# 7. 开始训练
trainer.train()

model.safetensors:   0%|          | 0.00/499M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] RobertaForSequenceClassification LOAD REPORT from: roberta-base
Key                        | Status     | 
---------------------------+------------+-
lm_head.dense.bias         | UNEXPECTED | 
lm_head.dense.weight       | UNEXPECTED | 
lm_head.layer_norm.bias    | UNEXPECTED | 
lm_head.layer_norm.weight  | UNEXPECTED | 
lm_head.bias               | UNEXPECTED | 
classifier.dense.bias      | MISSING    | 
classifier.dense.weight    | MISSING    | 
classifier.out_proj.weight | MISSING    | 
classifier.out_proj.bias   | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
[transformers] warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Epoch,Training Loss,Validation Loss,Accuracy
1,No log,0.424533,0.825459
2,0.461536,0.373803,0.839895
3,0.326054,0.425120,0.843832


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=1287, training_loss=0.37337922494172493, metrics={'train_runtime': 64.9806, 'train_samples_per_second': 316.294, 'train_steps_per_second': 19.806, 'total_flos': 1351930380203520.0, 'train_loss': 0.37337922494172493, 'epoch': 3.0})

In [4]:
# 8. 测试集预测与提交
test_df = pd.read_csv('data/test.csv')
test_ids = test_df['id']
test_df_processed = preprocess_df(test_df)
test_dataset = Dataset.from_pandas(test_df_processed)
tokenized_test = test_dataset.map(tokenize_function, batched=True)

predictions = trainer.predict(tokenized_test)
preds = np.argmax(predictions.predictions, axis=-1)

submission = pd.DataFrame({'id': test_ids, 'target': preds})
os.makedirs('outputs', exist_ok=True)
submission.to_csv('outputs/submission_roberta.csv', index=False)
print("DeBERTa Submission saved successfully!")

Map:   0%|          | 0/3263 [00:00<?, ? examples/s]

DeBERTa Submission saved successfully!
